In [2]:

import sys
sys.path.append('/home/pavel/pyadi-iio/')
sys.path.append('/home/pavel/libiio')
import adi
import matplotlib.pyplot as plt
import numpy as np

# Create radio
sdr = adi.ad9363("ip:192.168.2.1")
samp_rate = 20e6

'''Configure Rx properties'''
sdr.rx_enabled_channels = [0, 1]
sdr.sample_rate = int(samp_rate)
sdr.rx_rf_bandwidth = int(2e6)
sdr.rx_lo = int(2.71e9)
sdr.rx_buffer_size = 8192
sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"
sdr.rx_hardwaregain_chan0 = int(40)
sdr.rx_hardwaregain_chan1 = int(40)
samples = sdr.rx() # receive samples off Pluto
# while 1:
#     x = sdr.rx()
#     a = np.real(x[0][:])
#     b = np.real(x[1][:])
#     corr = np.correlate(np.real(a), np.real(b), mode='full')
#     plt.clf()
#     plt.plot(corr)
#     # plt.ylim([-20e6, 20e6])
#     plt.draw()
#     plt.pause(0.05)
#     print(np.argmax(corr))
#     plt.show()

In [ ]:
# Same pilot sequence as transmitter (must match for channel estimation)
def m_sequence(length=63, taps=(6, 5)):
    n = int(np.log2(length + 1))
    reg = np.zeros(n, dtype=int)
    reg[0] = 1
    out = []
    for _ in range(length):
        out.append(reg[-1])
        feedback = reg[taps[0] - 1] ^ reg[taps[1] - 1]
        reg[1:] = reg[:-1]
        reg[0] = feedback
    return np.array(out)

def pilot_qpsk_symbols(num_pilot_symbols=128, seed=0):
    np.random.seed(seed)
    bits_i = m_sequence(63)
    bits_q = np.roll(m_sequence(63), 17)
    n = num_pilot_symbols
    idx = np.arange(n) % 63
    x_int = (bits_i[idx] * 2 + bits_q[idx]).astype(int)
    x_degrees = x_int * 360 / 4.0 + 45
    x_radians = x_degrees * np.pi / 180.0
    return np.cos(x_radians) + 1j * np.sin(x_radians)

num_pilot_symbols = 128
samples_per_symbol = 16
pilot_symbols = pilot_qpsk_symbols(num_pilot_symbols)

In [ ]:
def get_received_pilots(rx_signal, pilot_symbols, samples_per_symbol=16):
    """
    Sync to transmitted pilot and return the received pilot symbols (same pilots as tx, as received).
    Returns: rx_pilot (received pilot symbols), delay_samp (pilot start in samples), rx_symbols (full symbol-rate stream).
    """
    n_pilot = len(pilot_symbols)
    n_syms = len(rx_signal) // samples_per_symbol
    rx_symbols = np.array([
        np.mean(rx_signal[k * samples_per_symbol:(k + 1) * samples_per_symbol])
        for k in range(n_syms)
    ], dtype=np.complex64)
    # Find pilot start via correlation with known transmitted pilot
    corr = np.correlate(rx_symbols, np.conj(pilot_symbols), mode='valid')
    delay_sym = np.argmax(np.abs(corr))
    delay_samp = delay_sym * samples_per_symbol
    # Extract received pilot (same length/position as transmitted pilot)
    rx_pilot = rx_symbols[delay_sym:delay_sym + n_pilot].copy()
    return rx_pilot, delay_samp, rx_symbols


def estimate_channel(rx_signal, pilot_symbols, samples_per_symbol=16):
    """
    Estimate channel from received signal using known pilot.
    Returns: delay (samples), channel impulse response (symbol-spaced), and frequency response.
    """
    rx_pilot, delay_samp, _ = get_received_pilots(rx_signal, pilot_symbols, samples_per_symbol)
    n_pilot = len(pilot_symbols)
    # Least-squares channel (per-symbol gain): H_k = Y_k / X_k
    with np.errstate(divide='ignore', invalid='ignore'):
        h_symbols = np.where(np.abs(pilot_symbols) > 1e-6,
                             rx_pilot / pilot_symbols, 0)
    h_avg = np.mean(h_symbols[np.abs(pilot_symbols) > 0.1])
    H_freq = np.fft.fft(h_symbols)
    return delay_samp, h_symbols, h_avg, H_freq

In [ ]:
# Receive and extract the same pilots (transmitted pilot sequence as received)
# Ensure rx center_freq and sample_rate match the transmitter
x = sdr.rx()

# Get received pilots for both channels (same pilots as tx, after channel)
rx_pilot_ch0, delay_samp, rx_symbols_ch0 = get_received_pilots(
    x[0][:], pilot_symbols, samples_per_symbol
)
rx_pilot_ch1, _, rx_symbols_ch1 = get_received_pilots(
    x[1][:], pilot_symbols, samples_per_symbol
)

print(f"Pilot delay: {delay_samp} samples ({delay_samp // samples_per_symbol} symbols)")
print(f"Received pilot ch0: {len(rx_pilot_ch0)} symbols (same as transmitted {num_pilot_symbols})")
print(f"Received pilot ch1: {len(rx_pilot_ch1)} symbols")

# Optional: compare transmitted vs received (scatter)
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(np.real(pilot_symbols), np.imag(pilot_symbols), alpha=0.5, label='Tx pilot', s=10)
ax.scatter(np.real(rx_pilot_ch0), np.imag(rx_pilot_ch0), alpha=0.5, label='Rx pilot ch0', s=10)
ax.set_xlabel('I'); ax.set_ylabel('Q'); ax.legend(); ax.set_title('Tx vs Rx pilots (same sequence)')
ax.axis('equal'); plt.tight_layout(); plt.show()

In [ ]:
# Plot received signal (all) — run after receive + extract pilots
n_show = min(400, len(x[0]))
sr = samp_rate
t_us = np.arange(n_show) / sr * 1e6

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1) Time domain: raw received I/Q (ch0)
axes[0, 0].plot(t_us, np.real(x[0][:n_show]), label='I', alpha=0.8)
axes[0, 0].plot(t_us, np.imag(x[0][:n_show]), label='Q', alpha=0.8)
axes[0, 0].set_xlabel('Time (µs)'); axes[0, 0].set_ylabel('Amplitude'); axes[0, 0].legend(); axes[0, 0].set_title('Received signal ch0 (raw baseband)')
axes[0, 0].grid(True, alpha=0.3)

# 2) Tx vs Rx pilots (constellation) — same pilots, ch0 and ch1
axes[0, 1].scatter(np.real(pilot_symbols), np.imag(pilot_symbols), alpha=0.4, s=15, label='Tx pilot', c='C0')
axes[0, 1].scatter(np.real(rx_pilot_ch0), np.imag(rx_pilot_ch0), alpha=0.5, s=15, label='Rx pilot ch0', c='C1')
axes[0, 1].scatter(np.real(rx_pilot_ch1), np.imag(rx_pilot_ch1), alpha=0.5, s=15, label='Rx pilot ch1', c='C2')
axes[0, 1].set_xlabel('I'); axes[0, 1].set_ylabel('Q'); axes[0, 1].legend(); axes[0, 1].set_title('Tx vs Rx pilots (same sequence)'); axes[0, 1].axis('equal'); axes[0, 1].grid(True, alpha=0.3)

# 3) Received spectrum (ch0)
N = len(x[0])
spec = np.fft.fftshift(np.fft.fft(x[0][:]))
f_axis = np.fft.fftshift(np.fft.fftfreq(N, 1 / sr)) / 1e6
axes[1, 0].plot(f_axis, 20 * np.log10(np.abs(spec) + 1e-10))
axes[1, 0].set_xlabel('Frequency (MHz)'); axes[1, 0].set_ylabel('Magnitude (dB)'); axes[1, 0].set_title('Received spectrum ch0 (baseband)')
axes[1, 0].grid(True, alpha=0.3)

# 4) Received pilot constellation only (ch0)
axes[1, 1].scatter(np.real(rx_pilot_ch0), np.imag(rx_pilot_ch0), alpha=0.6, s=25)
axes[1, 1].set_xlabel('I'); axes[1, 1].set_ylabel('Q'); axes[1, 1].set_title('Received pilot constellation ch0 (QPSK after channel)'); axes[1, 1].axis('equal'); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Channel estimation from received pilots (optional)
delay_samp, h_symbols, h_avg, H_freq = estimate_channel(
    x[0][:], pilot_symbols, samples_per_symbol
)
print(f"Average channel gain: {h_avg:.4f} (mag={np.abs(h_avg):.4f}, phase={np.angle(h_avg):.3f} rad)")
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(np.abs(h_symbols), '.-', label='|H| per symbol')
axes[0].set_ylabel('|H|'); axes[0].legend(); axes[0].set_title('Channel estimate (symbol-spaced)')
axes[1].plot(np.fft.fftshift(np.fft.fftfreq(len(H_freq))), np.fft.fftshift(np.abs(H_freq)))
axes[1].set_xlabel('Normalized frequency'); axes[1].set_ylabel('|H(f)|'); axes[1].set_title('Channel frequency response')
plt.tight_layout(); plt.show()

In [2]:
x = sdr.rx()

In [3]:
x[1].shape

(8192,)

In [4]:
sample_rate = 1e6 # Hz
center_freq = 915e6 # Hz

sdr.tx_enabled_channels = [0, 1]
sdr.sample_rate = int(samp_rate)
# sdr.tx_rf_bandwidth = int(2e6)
# sdr.tx_lo = int(2.71e9)

# sdr.gain_control_mode_chan0 = "manual"
# sdr.gain_control_mode_chan1 = "manual"
# sdr.tx_hardwaregain_chan0 = int(40)
# sdr.tx_hardwaregain_chan1 = int(40)
# samples = sdr.rx()

# N = 10000 # number of samples to transmit at once
# t = np.arange(N)/sample_rate
# samples = 0.5*np.exp(2.0j*np.pi*100e3*t) # Simulate a sinusoid of 100 kHz, so it should show up at 915.1 MHz at the receiver
# samples *= 2**14 # The PlutoSDR expects samples to be between -2^14 and +2^14, not -1 and +1 like some SDRs

# # Transmit our batch of samples 100 times, so it should be 1 second worth of samples total, if USB can keep up
# for i in range(100):
#     sdr.tx(samples) # transmit the batch of samples once